In [1]:
from git2kg import Git2KG
from git2kg.export import NetworkXExporter, PathPyGExporter, PyGExporter
from git2kg.labels import DeveloperLeaverLabelExtractor

with open("../github_token.githubtoken") as f:
    token = f.read()

## Mining a repository

`git2KG` provides a simple interface for mining a repository hosted on GitHub. You simply need to provide the URL of the GitHub repository and a GitHub access token, with the priviledge `public:repo`. If you like to persist the loaded dataframes during mining, you are welcome to path a directory, that is used to create an instance of the `FilePersistenceManager` class. The resulting object is of type `Git2KG`, the main class of the package, which is used to globally control the mined data and the data extraction process. Depending on the dataset you're mining this may take a while, progress bars to overview the mining process are supplied.

In [2]:
# Mine the repository of the `pathpyG` package, a frame work for GPU-accelerated Next-Generation network analytics and graph learning
xzG2kg = Git2KG.from_github_url("https://github.com/tukaani-project/xz", token, persistence_path="persist", processes=8)

# Save the mined data to the directory `mined-repos`
xzG2kg.save("mined-repos")

Loading parallely on 8 processes: 0it [00:00, ?it/s]
Loading parallely on 8 processes: 0it [00:00, ?it/s]
Loading parallely on 8 processes: 0it [00:00, ?it/s]
author identity disambiguation: 100%|██████████| 3/3 [00:00<00:00, 3526.60it/s]


'mined-repos/tukaani-project_xz-git2kg.tar.gz'

When printing the mined `Git2KG` instance, we get an overview of the data it's holding, and the metadata, that will be used for data extraction

In [3]:
print(xzG2kg)


Git2KG
------
Name: 	 	 tukaani-project_xz
Mined On: 	 2026-11-10T08:52:25.797343
Frequency: 	 ME

GitData
-------
3142 Commits
447650 Edits
252 Issues
1131 Issue Comments
236 Events
244 Reviews
        
        


Now, that we have saved the mined data, we can load it again. By printing the loaded instance, we can confirm, that the loaded data is identical to the mined data 

In [4]:
xzG2kg = Git2KG.load("mined-repos/tukaani-project_xz-git2kg.tar.gz")
print(xzG2kg)


Git2KG
------
Name: 	 	 tukaani-project_xz
Mined On: 	 2026-11-10T08:52:25.797343
Frequency: 	 ME

GitData
-------
3142 Commits
447650 Edits
252 Issues
1131 Issue Comments
236 Events
244 Reviews
        
        


`git2KG` is build to extract temporal knowledge graph snapshots from the given data. How the snapshots are constructed, depends on the `freq` parameter, that can be set on the `Git2KG` object. It controls the frequency of the knowledge graph sequence. Valid values are strings that comply with `pandas`' date tim offset aliases (https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases). Each snapshot holds the interactions of its own timeslice, while node features describe the state each node has reached by the end of that timeslice. If you wish to analyze a static knowledge graph, set `Git2KG.freq = None`. This will return a single snapshot, containing inferred from the global data handled by the `Git2KG` object.

To extract snapshots, call the `get_snapshots` method on the `Git2KG` object. You can pass a class reference for each of the `LabelExtractor` subclasses.

In [5]:
xzG2kg.freq = "3ME"
snapshots = xzG2kg.get_snapshots(actor_label_extractor=DeveloperLeaverLabelExtractor)

Compiling feature matrices for snapshots: 100%|██████████| 77/77 [00:08<00:00,  8.79it/s]


Since we want to use popular and feature-rich network analysis tool, we are exporting the snapshots created by the given frequency (month-end (ME) frequency in our case), to one of those frameworks. `git2KG` provides three build in exporters: `PathPyGExporter`, which produces snapshots in the `torch_geometric.data.HeteroData` format, `PathPyGExporter`, which returns each snapshot as a `pathpyG.Graph` object, and the `NetworkXExporter` providing the export of snapshots as `networkx.MultiDiGraph` objects. The `export_snapshots` method of each exporter accepts a `NetworkSnapshots` object, as returned by the `get_snapshots` method of `Git2KG` and returns the snapshots in a dictionary, keyed by the upper-bound timestamp of the corresponding temporal snapshot. All three provided exporters, retain all global nodes across all snapshots. If you wish for the snapshots to contain only nodes that are source or destination of an edge, please use the provided methods for removing isolated nodes in the corresponding framework.

If you wish to export the snapshots to different format, you are encouraged to write your own implementation of the abstract class `NetworkExporter` by implementing the `do_export` method.

In [6]:
pyG_exporter = PyGExporter()
pathpyG_exporter = PathPyGExporter()
networkX_exporter = NetworkXExporter()

pyg_snapshots = pyG_exporter.export_snapshots(snapshots)
pathpyG_snapshots = pathpyG_exporter.export_snapshots(snapshots)
networkX_snapshots = networkX_exporter.export_snapshots(snapshots)

print("------- PyG Hetero data export -------")
print((pyg_snapshots))
print("-------- PathPyG Graph export --------")
print(pathpyG_snapshots)
print("---- NetworkX MultiDiGraph export ----")
print(networkX_snapshots)


Exporting Snapshots: 100%|██████████| 77/77 [00:04<00:00, 17.29it/s]

------- PyG Hetero data export -------
{Timestamp('2007-12-31 22:42:33+0000', tz='UTC'): HeteroData(
  actor={
    x=[345, 25],
    y=[345],
    active_mask=[345],
  },
  issue={
    x=[252, 8],
    active_mask=[252],
  },
  commit={
    x=[3142, 9],
    active_mask=[3142],
  },
  file={
    x=[745, 8],
    active_mask=[745],
  },
  (actor, edit_add, actor)={
    edge_index=[2, 1],
    edge_weight=[1],
  },
  (actor, edit_delete, actor)={
    edge_index=[2, 2],
    edge_weight=[2],
  },
  (actor, issue_comment, actor)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, review_request, actor)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, review, actor)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, assign_issue, actor)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, mentions, actor)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, create_issue, issue)={
    edge_index=[2, 0],
    edge_weight=[0],
  },
  (actor, comment_issue